In [1]:
from science_jubilee.JubileeManager import JubileeManager
from science_jubilee.decks.LoadAll import load_all
from science_jubilee.tools.Loop import *
from science_jubilee.labware.Labware import *
from science_jubilee.decks.Deck import *

In [2]:
# Connexion à la Jubilee
jm = JubileeManager(address="10.0.9.6", simulated=False)
ino = Loop


2026-03-06 16:22:12 - [JubileeController]  - INFO - Initializing JubileeController (simulated=False, address=10.0.9.6)
2026-03-06 16:22:12 - [JubileeController]  - WARNING - Disconnecting this application from the network will halt connection to Jubilee.
2026-03-06 16:22:12 - [JubileeController]  - INFO - Connecting to Jubilee machine...
2026-03-06 16:22:13 - [JubileeController]  - INFO - Successfully connected and initialized Jubilee machine.
2026-03-06 16:22:13 - [JubileeManager]     - INFO - JubileeManager initialized (simulated=False, max_tools=4).


In [3]:
# Homing
jm.controller.home_all()

Is a tool currently mounted? [y/n]  n
Is the deck clear of any obstacles? [y/n]  y


2026-03-06 15:16:30 - [JubileeController]  - INFO - All axes homed successfully.


In [3]:
# Descendre le plateau pour positionner le deck
jm.controller.move_to(z=300)
jm.controller.move_to(x=150, y=150)

In [4]:
# Enregistrer le deck
deck = jm.load_deck("test1")
load_all(jm,"test1")

2026-03-06 16:23:50 - [Deck]               - INFO - Loading deck configuration from: D:\Polytech\2025-2026\projet_indus\science-jubilee\src\science_jubilee\decks\deck_definition\test1.json
2026-03-06 16:23:50 - [Deck]               - INFO - Deck 'Experience1' loaded with 1 slots.
2026-03-06 16:23:50 - [JubileeManager]     - INFO - Deck 'Experience1' loaded from 'D:\Polytech\2025-2026\projet_indus\science-jubilee\src\science_jubilee\decks\deck_definition'.
2026-03-06 16:23:50 - [Tool]               - INFO - Tool 'Inoculator' (index 0) initialized.
2026-03-06 16:23:50 - [JubileeManager]     - INFO - Tool 'Inoculator' loaded at index 0.
2026-03-06 16:23:50 - [JubileeManager]     - INFO - Offset for tool at index 0 set to (0, -43.5, 0).


In [5]:
#affiche les slots
jm.deck.list_slots()
jm.deck.get_all_slot_machine_coordinates()

{'0': (148.12, 83.59)}

In [6]:
jm.status()

{'deck': 'Experience1',
 'tools': [{'index': 0, 'name': 'Inoculator'}],
 'active_tool_index': None,
 'active_tool_name': None,
 'tool_offsets': {0: (0, -43.5, 0)}}

In [7]:
jm.controller.move_to(x=150, y=150)
jm.controller.move_to(z=50)



In [8]:
# 1. On récupère les infos de ton outil par son nom
dict_outil = jm.get_tool_by_name("Inoculator")
idx = dict_outil['index']

# 2. On prépare l'outil physiquement
jm.controller.pickup_tool_sequence(idx)
jm.set_active_tool(idx)
jm.set_tool_offset(idx, (0, -31.5, 35))

tool2 = jm.tools_list[idx]
tool2.__class__ = Loop
tool2._machine = jm.controller  # On le lie au contrôleur pour les mouvements
tool2.is_active_tool = True     # On valide le décorateur de sécurité

# 4. CORRECTIF : On crée la méthode safe_z_movement si elle manque
if not hasattr(jm.controller, 'safe_z_movement'):
    # On remonte à 80mm de hauteur pour ne rien toucher
    tool2.safe_z_movement = lambda: jm.controller.move_to(z=80)
    print("✅ Correctif safe_z_movement appliqué.")

print(f"✅ Outil '{tool2.name}' prêt (Classe: {type(tool2).__name__})")

2026-03-06 16:24:16 - [JubileeController]  - INFO - Starting pickup_tool sequence for tool 0.
2026-03-06 16:24:23 - [JubileeController]  - INFO - pickup_tool_sequence completed for tool 0.
2026-03-06 16:24:23 - [JubileeManager]     - INFO - Tool at index 0 set as active tool.
2026-03-06 16:24:23 - [JubileeManager]     - INFO - Offset for tool at index 0 set to (0, -31.5, 35).


✅ Correctif safe_z_movement appliqué.
✅ Outil 'Inoculator' prêt (Classe: Loop)


In [9]:
# On s'assure d'être en hauteur avant de commencer
jm.controller.move_to(z=80)

rows = ['A', 'B', 'C', 'D']
cols = range(1, 7)

# On définit la SOURCE fixe (ici slot A1 du slot 
slot_0 = jm.deck.get_slot('0')
well_source = slot_0.labware.wells['A1']

print(f"🚀 Début de la distribution depuis {well_source.name}...")

# --- LA BOUCLE DE DISTRIBUTION ---
for r in rows:
    for c in cols:
        well_dest_name = f"{r}{c}" # Génère A1, A2... D6
        
        # On récupère l'objet Well de destination
        well_dest = slot_0.labware.wells[well_dest_name]
        
        # Optionnel : On évite de transférer A1 vers lui-même
        if well_dest_name == 'A1':
            continue

        print(f"Transfert de {well_source.name} vers {well_dest_name}...")
        
        try:
            # Appel du transfert (la sécurité safe_z est déjà injectée)
            tool2.transfer(
                source=well_source,      # FIXE
                destination=well_dest,   # CHANGE à chaque itération
                s=2000,
                sweep_x=3,
                sweep_y=3,
                sweep_z=10,
                sweep_speed=150,
                up_speed=800,
                randomize_pickup=False
            )
        except Exception as e:
            print(f"❌ Erreur critique au puits {well_dest_name}: {e}")
            # Sécurité : on remonte immédiatement
            jm.controller.move_to(z=80)
            break 

print("✅ Distribution terminée sur toute la plaque !")

🚀 Début de la distribution depuis A1...
Transfert de A1 vers A2...
Transfert de A1 vers A3...
Transfert de A1 vers A4...
Transfert de A1 vers A5...
Transfert de A1 vers A6...
Transfert de A1 vers B1...
Transfert de A1 vers B2...



KeyboardInterrupt



In [18]:
print(well_source)
print(slot)
print(slot_0.labware.wells['A1'])

Well A1 at coordinates (15.0, 15.0, 2.5)
Slot(slot_index='0', coordinates=(148.12, 83.59), shape='rectangle', width=127.76, length=85.48, diameter=None, has_labware=True, labware=wellPlate: greiner_24_wellplate_3300ul)


AttributeError: 'Slot' object has no attribute 'wells'